# Capítulo 5 — Classificação

**Livro:** começa na página **226** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
rng = np.random.default_rng(42)
print("Pronto. pandas", pd.__version__)

## A Solução Naive · PDF p. 229

#### 🧪 Exercício: Naive Bayes gaussiano

Com os dados de câncer de mama (`load_breast_cancer`), separe 30% para teste e treine `GaussianNB`. Mostre a acurácia de teste.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(round(GaussianNB().fit(Xtr, ytr).score(Xte, yte), 3))

## Discriminante Linear de Fisher · PDF p. 234

#### 🧪 Exercício: Análise discriminante linear

No mesmo problema, treine `LinearDiscriminantAnalysis` e compare a acurácia de teste com o Naive Bayes.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(round(LinearDiscriminantAnalysis().fit(Xtr, ytr).score(Xte, yte), 3))

## Interpretando os Coeficientes e as Razões de Chances · PDF p. 243

#### 🧪 Exercício: Logística e razão de chances

Simule a inadimplência com P = logística(−3 + 0,8·dívida/renda), dívida/renda de 0 a 8. Ajuste `smf.logit` e interprete `exp(coef)` como razão de chances.

In [ ]:
d = pd.DataFrame({"divida_renda": rng.uniform(0, 8, 2000)})
d["inad"] = (rng.random(2000) < 1 / (1 + np.exp(-(-3 + 0.8 * d.divida_renda)))).astype(int)
m = smf.logit("inad ~ divida_renda", d).fit(disp=0)
print(m.params.round(3), np.exp(m.params["divida_renda"]).round(3))  # cada +1 multiplica as chances por ~2,2

## Matriz de Confusão · PDF p. 251

#### 🧪 Exercício: Matriz de confusão e métricas

Com uma regressão logística nos dados de câncer de mama (variáveis padronizadas), mostre a matriz de confusão e o `classification_report` no teste.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
print(confusion_matrix(yte, m.predict(Xte))); print(classification_report(yte, m.predict(Xte), digits=3))

## Curva ROC · PDF p. 255

#### 🧪 Exercício: Curva ROC e AUC

Desenhe a curva ROC da regressão logística do exercício anterior e calcule a AUC (`roc_curve`, `roc_auc_score`).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
p = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
fpr, tpr, _ = roc_curve(yte, p); plt.plot(fpr, tpr); plt.plot([0, 1], [0, 1], "--"); plt.show()
print(round(roc_auc_score(yte, p), 4))

## Lift · PDF p. 258

#### 🧪 Exercício: Gráfico de lift por decil

Ordene os clientes de teste pela probabilidade prevista, divida em 10 decis e calcule o *lift* de cada decil (taxa de positivos no decil ÷ taxa geral).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X, y = load_breast_cancer(return_X_y=True)
y = 1 - y  # positivo = maligno
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
p = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
d = pd.DataFrame({"p": p, "y": yte}); d["decil"] = pd.qcut(d.p.rank(method="first", ascending=False), 10, labels=range(1, 11))
print((d.groupby("decil", observed=True).y.mean() / d.y.mean()).round(2))

## Oversampling e Ponderação Acima/Abaixo · PDF p. 263

#### 🧪 Exercício: Classe rara: ponderação

Gere um problema com 2% de positivos (`make_classification(weights=[0.98])`). Compare a revocação dos positivos de uma logística comum com uma com `class_weight="balanced"`.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score
X, y = make_classification(5000, weights=[0.98], random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
for cw in (None, "balanced"):
    pr = LogisticRegression(class_weight=cw, max_iter=1000).fit(Xtr, ytr).predict(Xte)
    print(cw, "revocação", round(recall_score(yte, pr), 3), "precisão", round(precision_score(yte, pr), 3))